# Applied Deep Learning Foundations, Modality Architectures & System Design

## Overview
This notebook serves as a structured theoretical reference and system design guide covering the core paradigms of applied deep learning across multiple modalities (Computer Vision, Natural Language Processing, Multimodal AI, Tabular Learning, and Collaborative Filtering), alongside real-world deployment trade-offs, the Drivetrain Approach for actionable data products, and the FastAI library ecosystem.

## Key Technical Components
* **Computer Vision Paradigms:** Detailed taxonomy of CV tasks (Recognition, Object Detection, Semantic Segmentation), domain generalization issues, out-of-domain data drift, data augmentation strategies, and cross-modality transformations (e.g., audio spectrograms as visual inputs).
* **NLP Fluency vs. Factuality:** Analysis of modern text generation systems, stylistic imitation, hallucination risks, misinformation dynamics ("generator vs. detector" arms race), and utility-accuracy trade-offs in downstream production tasks (translation, summarization, entity recognition).
* **Multimodal Systems & Human-in-the-Loop:** Hybrid workflows combining predictive/generative deep learning models with expert human verification to achieve high throughput without sacrificing accuracy (e.g., time-critical clinical triage).
* **Tabular Deep Learning & Categorical Embeddings:** Deep learning integration alongside ensemble trees (GBMs, Random Forests), accelerating data pipelines with GPU tooling (RAPIDS), and mapping high-cardinality discrete categorical variables into continuous latent embeddings.
* **Recommendation Engines & Latent Factor Modeling:** Collaborative filtering architectures over sparse user-item interaction matrices, handling cardinality, and addressing the "helpfulness vs. preference" problem in user recommendations.
* **The Drivetrain Approach (Actionable Product Design):** A 4-step framework (Objective $\rightarrow$ Levers $\rightarrow$ Data $\rightarrow$ Models) to build causal, decision-driving AI systems and counter recommendation blindspots using dual-model counterfactual evaluation strategies.
* **FastAI Ecosystem Architecture:** Modular abstraction breakdown across `fastai` (high-level learner API), `fastcore` (low-level idiomatic Python extensions), and `fastbook` (pedagogical reference and utilities).

------------
------------
------------

# THE STATE OF DEEP LEARNING


## Computer Vision

(Big 3 of CV)
- **Object Recognition**: Computers can recognize what items are in the image.
- **Object Detection**: Computers recognize where objects are in an image and can highlight their location and name each found object.
- **Segmentation**: Every pixel is categorized based on what kind of object it is part of.

Deep learning isn't good at recognizing images that are significantly different in structure or style to those used to train the model.
E.g. no black-and-white/hand-drawn images in training set may lead the model to perform poorly on black-and-white/hand-drawn images.
- There is no general way to check what types of images are missing in our training set, but there are some ways to try to recognize when unexpected image types arise in the data when the model is being used in production(this is known as checking out for ***out-of-domain*** data.

- **Data Augmentation**: One of the hardest challenges in CV is image labelling. One approach that is particularly helpful is to synthetically generate variations of input images, such as by rotating them or changing their brightness and contrast; this is called data augmentation and also works well for text and other types of model.

- Sometimes a problem doesn't seem to be a CV problem but it might be possible to turn it into one. E.g. while classifying sounds we can convert the sounds into images of their acoustic waveforms and then training a model on those images. 

## Text(Natural Language Processing)

DL in NLP has achieved **human-level fluency** but not achieved **human-level accuracy**.

* Mastery of Form and Style:
- Computers are very good at classifying both short and long documents based on categories such as spam or not spam, sentiment(positive/negative), author, website etc.
- DL is also very good at generating context-appropriate text, such as replies to social media posts or imitating a particular aurhor's style.

* The "Truth" Gap - A Critical Danger:
- Generated responses by AI aren't factually correct like medical records. To a layman it may appear convincing, contextually correct but it may not be actually correct.

* Societal Risks and AI Arms Race:
- Mass Scale Disinformation can be easily created like "troll farms" at a much larger scale.
- Detection Deficit: If let's say we built a "detector AI" to detect "fake text". Then a generator AI can be trained on the detector AI to generate text until Detector can't catch it anymore. (Generator outperforming Detector).

* Utility vs. Reliability:
- Despite risk of "hallucinations"(incorrect information) DL powers most common tools for translation, document summarization, and entit recognition(e.g: Google translate). We have to trade speed and quality for accuracy(output maybe factually wrong) for these generations.


## Combining text and images

DL can perform surprisingly well here also, but the only concern is accuracy.
So, if we combine both human interaction along with DL model's work the its the best of two world's. It would be orders of magnitude more productive than manual methods and accurate as well unlike entirely automated process. 

- E.g: An automatic system from CT scans identify stroke victims and send a high priority alert(as there is only a 3 hr window to treat strokes). At same time all scans continue to sent to Radiologists usual way so there is no reduction in human input. AI can also analyze reports and warn them that they may have missed to be extra sure.

## Time Series and Tabular Data

- DL is rarely used in isolation in tabular data. It usually used in "ensemble" of other models like - Random Forests or Gradient Boosting Machines(GBMs). If we're already using these traditional tools then DL might only offer incremental improvements rather than a massive breakthrough.
- The primary advantage of DL comes with "messy" or complex data like - columns containing natural language(books, titles, etc.), and high - cardinality categorical columns(i.e., something that contains a large number of discrete choices like zip code or product ID).
- DL models generally takes longer to tran than random forests/gradient boosting machines. But this speed gap is narrowing doe to libraries like **RAPIDS**- which use GPU acceleration to speed up the entire data pipeline. 

## Recommendation Systems

Recommendation Systems are just a special type of tabular data. In particular they generally have a high-cardinality categorical variable representing users, and another one representing products(or similar).
- E.g in a company like Amazon, every purchase(column) that have been made by the customer(row) is represented in a giant sparse matrix.
- Once they have the data in the format, data scientists apply **Collaborative Filtering** to fill the matrix.
- If customer A buys item 1 and 10, while customer B buys 1,2,4,10, the recommendation engine will suggest item 2,4 to customer A. DL systems are generally good at handling high-cardinality categorical variables.
- The "Helpfulness" Problem: A major flaw in these systems is that they predict preference, not utility. An AI might correctly guess you like an author, but it only recommends books you already own or know about, the recommendation is "correct" but useless.


* Cardinality refers to the number of unique values in a column.
  E.g: Low-cardinality - Gender, Boolean(True/False)
       High-cardinality - A column that has hundreds, thousands or millions of unique possibilities(like Amazon example).
* Collaborative Filtering is based on simple human intuition: "If you like what I liked in the past, you'll probably like what I like in the future." {Amazon recommendation example in the Sparse matrix(where rows are mostly 0s, Collaborative filtering fills blank on based of what the user hasn't seen which other likely user seen)}
* Embedding is a way of turning a "thing" (like a word, a user, or a product) into a list of numbers that represents the hidden characteristics, allowing the computer to map similar things close together in a mathematical space. E.g instead of saying "User#123", the embedding might be [0.9,-0.2,0.5] where 0.9 represents how much action, -0.2 how much romance, 0.5 how much budget to recommend better movies according to taste.


### Other Data Types
- Domain-specific data types fit nicely into existing categories. Like, sounds can be represented as spectrograms, which can be treated as images; standard DL approaches for images work on them really well. Another example NLP in protein-analysis.

## Drivetrain Approach
- Step 1: Define the objective (What am I trying to achieve?) - like building recommendation engine
- Step 2: Levers(what inputs can we control) - like ranking of recommendations 
- Step 3: Data(what data can we collect) - like more data/new data collected after recommendation (require consucting random experiments)
- Step 4: Models(how the levers influence the objective) - like the final recommendation engine

This above approach helps eradicating "Terry Pratchet" problem(recommeding things the user already knows about).

* "Two Model Strategy":
  Build 2 models: Model A(Prob user buys through recommendation); Model B(Prob user buys without recommendation)
  Model B - Model A :
      If result High: recommendation caused the sale
      If result Zero/Low: recommendation is waste of space
  

# fastai, fastcore and fastbook

1. **`fastai(The Library)`**: This is high-level DL library that we actually use to build models. It's built on PyTorch and designed to achieve state-of-the art results in very few lines of code.
   * It's main purpose is to train models for vision, text, tabular data, and more. The key tool is the "Learner" object.

2. **`fastcore(The Foundation)`**: This is a low-level library that the fast.ai team built to make Python "better" for their specific need. It's essentially a set of "power tools" for Python programmers.
   * It purpose is it provides advances features like better delegation, type dispatching (changing function behaviour based on input types), and more powerful lists/tuples.

3. **`fastbook(The Book & Helpers)`**: This refers to the book Deep Learning for Coders with fastai and PyTorch by Jeremy Howard and Sylvian Gugger.
   *  When we pip install fastbook, we download a small utility library that contains helper functions specificallly for the book's chapters(like setup_book()) to configure your environment). It also is a collection of Jupyter Notebooks that contain the entire text and code of the book.